In [0]:

def read_raw_data(path):
    df = (spark.read
        .option("delimiter",";")
        .option("header", "true")
        .option("escape", '"')
        .csv(path))
    return df

df_urbanistico = read_raw_data("/Volumes/workspace/raw/raw_empresas_recife/resources_39927a3d-3235-436a-9dc8-df7f4ea3b720_licenciamento_urbanistico.csv")   
df_ambiental = read_raw_data("/Volumes/workspace/raw/raw_empresas_recife/resources_982d6c85-906a-4ea8-90e0-31f55eac63f9_licenciamento_ambiental.csv")
df_sanitario = read_raw_data("/Volumes/workspace/raw/raw_empresas_recife/resources_d09900c8-14f2-4dc4-a128-b73a514ec791_licenciamento_sanitario.csv")


In [0]:
print(df_urbanistico.count(), len(df_urbanistico.columns))
print(df_ambiental.count(), len(df_ambiental.columns))
print(df_sanitario.count(), len(df_sanitario.columns))

In [0]:

print(df_urbanistico.filter(df_urbanistico.cnpj.isNotNull()).count())
print(df_ambiental.filter(df_ambiental.cnpj.isNotNull()).count())
print(df_sanitario.filter(df_sanitario.cnpj.isNotNull()).count())

In [0]:
from pyspark.sql.functions import countDistinct

df_urbanistico.select(countDistinct("cnpj")).show()
df_ambiental.select(countDistinct("cnpj")).show()
df_sanitario.select(countDistinct("cnpj")).show()

In [0]:
print(df_urbanistico.filter(df_urbanistico.cnpj == "").count())
print(df_ambiental.filter(df_ambiental.cnpj == "").count())
print(df_sanitario.filter(df_sanitario.cnpj == "").count())

In [0]:
from pyspark.sql.functions import col, length

print(df_urbanistico.filter(length(col("cnpj")) == 14).count())
print(df_ambiental.filter(length(col("cnpj")) == 14).count())
print(df_sanitario.filter(length(col("cnpj")) == 14).count())

In [0]:
print(df_urbanistico.filter(df_urbanistico.cnpj.rlike('^[0-9]{14}$')).count())
print(df_ambiental.filter(df_ambiental.cnpj.rlike('^[0-9]{14}$')).count())
print(df_sanitario.filter(df_sanitario.cnpj.rlike('^[0-9]{14}$')).count())

In [0]:
cnpj_urb = df_urbanistico.filter(df_urbanistico.cnpj.isNotNull()).select("cnpj").distinct()
cnpj_amb = df_ambiental.filter(df_ambiental.cnpj.isNotNull()).select("cnpj").distinct()
cnpj_sani = df_sanitario.filter(df_sanitario.cnpj.isNotNull()).select("cnpj").distinct()

print(cnpj_urb.join(cnpj_amb, "cnpj").count())
print(cnpj_urb.join(cnpj_sani, "cnpj").count())
print(cnpj_amb.join(cnpj_sani, "cnpj").count())

In [0]:
all_cnpj = cnpj_urb.union(cnpj_amb).union(cnpj_sani).distinct()
print(all_cnpj.count())

In [0]:
cnpj_join = cnpj_urb.join(cnpj_amb, "cnpj").join(cnpj_sani, "cnpj").count()
print(cnpj_join)

In [0]:
df_empresas_ativas = read_raw_data("/Volumes/workspace/raw/raw_empresas_recife/resources_6d5e72aa-3f4b-4dc3-817c-0e0305cef538_empresasativender.csv")
display(df_empresas_ativas)

In [0]:
print(df_empresas_ativas.count(), len(df_empresas_ativas.columns))
df_empresas_ativas.printSchema()

In [0]:
cnpj_empAT = df_empresas_ativas.filter(df_empresas_ativas.cnpj.isNotNull()).select("cnpj").distinct()
df_empresas_ativas.filter(~df_empresas_ativas.cnpj.rlike ('^[0-9]{14}$')).select("cnpj", "razao_social").show(20, truncate=False)

In [0]:
print(cnpj_empAT)

In [0]:
cross_cnpj = cnpj_empAT.join(all_cnpj, "cnpj").count()
print(cross_cnpj)

In [0]:
df_empresas_inativas = read_raw_data("/Volumes/workspace/raw/raw_empresas_recife/resources_e080550e-7238-444b-a821-8f5acfd82df2_empresasinativender.csv")
print(df_empresas_inativas.count(), len(df_empresas_inativas.columns))
df_empresas_inativas.printSchema()

In [0]:
set(df_empresas_ativas.columns) == set(df_empresas_inativas.columns)

In [0]:
df_empresas_ativas.unionByName(df_empresas_inativas).distinct().count()

In [0]:
df_empresas = df_empresas_ativas.unionByName(df_empresas_inativas)

print(df_empresas.count())                          
print(df_empresas_ativas.count() + df_empresas_inativas.count())  
print(df_empresas.select("cnpj").distinct().count())  

In [0]:
cnpj_emp_at   = df_empresas_ativas.select("cnpj").distinct()
cnpj_emp_inat = df_empresas_inativas.select("cnpj").distinct()

cnpj_emp_at.join(cnpj_emp_inat, "cnpj").count()

In [0]:
cnpj_emp_todos = df_empresas.select("cnpj").distinct()
all_cnpj.join(cnpj_emp_todos, "cnpj").count()

In [0]:
sem_match = all_cnpj.join(cnpj_emp_todos, "cnpj", "left_anti").count()
print(sem_match)

In [0]:
sem_match_cnpj = all_cnpj.join(cnpj_emp_todos, "cnpj", "left_anti")

(df_urbanistico
 .join(sem_match_cnpj, "cnpj")
 .select("cnpj", "razao_social", "bairro", "tipo_processo")
 .distinct()
 .show(30, truncate=False))

In [0]:
lic = (df_urbanistico.select("cnpj","razao_social")
       .unionByName(df_ambiental.select("cnpj","razao_social"))
       .unionByName(df_sanitario.select("cnpj","razao_social")))

info = lic.join(sem_match_cnpj, "cnpj").distinct()
info.count()

In [0]:
eh_pf     = col("razao_social").rlike(r" [0-9]{11}$")
eh_filial = substring("cnpj", 9, 4) != "0001"

print(info.filter(eh_pf).count())
print(info.filter(~eh_pf & eh_filial).count())
print(info.filter(~eh_pf & ~eh_filial).count())